<p style ="text-align:center">
    <img src="http://epecora.com.br/DataFiles/BannerUFPR.png" width="700" alt="PPGOLD/PPGMNE Python:INTRO"  />
</p>

# Data Science for Business

## Prof. Eduardo Pécora

## K-Fold Cross Validation
Tempo estimado: **30** minutos

## Objetivos

Após completar esta aula, você será capaz de:

* Separar os dados em dados de treino e teste
* Usar o K-Fold para avaliar a eficiência do modelo

## Bibliotecas

In [25]:
# importando a biblioteca pandas para manipulação de dados
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelBinarizer
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

# importando as bibliotecas do matplotlib para gerar gráficos
import matplotlib as mpl
import matplotlib.pyplot as plt

# Importando a biblioteca numpy e math que fornece funções matemáticas básicas
import numpy as np
import math 

# Importando biblioteca do seaborn para gerar gráficos mais atraentes e informativos
import seaborn as sns

# Importando a classe LinearRegression do sklearn 
# Essa classe implementa uma versão da regressão linear simples ou múltipla
# Usado para modelar a relação entre uma variável dependente contínua e uma ou mais variáveis independentes.
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Usado para exibir os gráficos gerados pela biblioteca Matplotlib diretamente no notebook, sem precisar abrir uma janela externa.
%matplotlib inline


## Obtendo dados

In [26]:
# Caminho do arquivo csv
caminho = "https://raw.githubusercontent.com/EduPekUfpr/PythonProject/refs/heads/main/Dados/MeuAutoLimpo.csv" 

#Obtendo arquivo e passando-o para um dataframe
df = pd.read_csv(caminho) 
df.head()

,symboling,normalized-losses,make,fuel-type,aspiration,num-of-doors,body-style,drive-wheels,engine-location,wheel-base,...,engine-size,fuel-system,bore,stroke,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,price
0,3,122,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111.0,5000.0,21,27,13495
1,3,122,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111.0,5000.0,21,27,16500
2,1,164,alfa-romero,gas,std,two,hatchback,rwd,front,94.5,...,152,mpfi,2.68,3.47,9.0,154.0,5000.0,19,26,16500
3,2,164,audi,gas,std,four,sedan,fwd,front,99.8,...,109,mpfi,3.19,3.40,10.0,102.0,5500.0,24,30,13950
4,2,164,audi,gas,std,four,sedan,4wd,front,99.4,...,136,mpfi,3.19,3.40,8.0,115.0,5500.0,18,22,17450


## Tranformação das variáveis

*Dúvidas? Veja a aula de Regressão Linear.*

In [27]:
df_dummy = df.copy()

label_binarizer = LabelBinarizer()
# https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelBinarizer.html

dicotomica = df_dummy['num-of-doors']
label_binarizer.fit(dicotomica)
df_dummy['num-of-doors'] = label_binarizer.transform(dicotomica)

# renomeando as marcas
df_dummy['make'] = df_dummy['make'].replace({'bmw':'BMW', 'doge': 'dodge', 'VW':'volkswagen', 'volv1':'volvo' })

# Crie uma instância do codificador OneHotEncoder
encoder = OneHotEncoder()

# Ajuste e transforme os dados da coluna "make"
poly_array = encoder.fit_transform(df_dummy[['make']])

# Crie um DataFrame Pandas com as variáveis dummy
poly_df = pd.DataFrame(poly_array.toarray(), columns=encoder.get_feature_names_out(['make']))

# Concatene o DataFrame dummy com os outros dados
df_dummy = pd.concat([df_dummy.drop('make', axis=1), poly_df], axis=1)

df_dummy.drop(['fuel-type', 'aspiration', 'body-style', 'drive-wheels', 
               'engine-location', 'engine-type', 'num-of-cylinders', 'fuel-system'], axis =1, inplace = True)
df_dummy.info()

<class 'pandas.DataFrame'>
RangeIndex: 201 entries, 0 to 200
Data columns (total 39 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   symboling           201 non-null    int64  
 1   normalized-losses   201 non-null    int64  
 2   num-of-doors        201 non-null    int64  
 3   wheel-base          201 non-null    float64
 4   length              201 non-null    float64
 5   width               201 non-null    float64
 6   height              201 non-null    float64
 7   curb-weight         201 non-null    int64  
 8   engine-size         201 non-null    int64  
 9   bore                201 non-null    float64
 10  stroke              201 non-null    float64
 11  compression-ratio   201 non-null    float64
 12  horsepower          201 non-null    float64
 13  peak-rpm            201 non-null    float64
 14  city-mpg            201 non-null    int64  
 15  highway-mpg         201 non-null    int64  
 16  price              

## Variáveis Dependentes e Independentes

In [28]:
# Separate features and target variable
X = df_dummy.drop(columns=['price'])
y = df_dummy['price']

## K-Fold CrossValidation

In [39]:
# Initialize the KFold object with 10 folds
kf = KFold(n_splits=10, shuffle=True, random_state=42)

## Avalia o modelo nas variáveis de treino e teste

In [40]:
scores_manual = []
scores2_manual = []

for train_index, test_index in kf.split(X):

    X_train = X.iloc[train_index]
    X_test  = X.iloc[test_index]

    y_train = y.iloc[train_index]
    y_test  = y.iloc[test_index]

    lm_fold = LinearRegression()
    lm_fold.fit(X_train, y_train)

    y_pred = lm_fold.predict(X_test)
    scores_manual.append(r2_score(y_test, y_pred))
    
scores_manual = np.array(scores_manual)
print("Manual:", scores_manual)

Manual: [0.93113091 0.94750352 0.73564664 0.78433343 0.88053631 0.94724034
 0.81643834 0.86305748 0.71482555 0.86833645]


## Um código mais compacto

In [48]:
scores_cv = cross_val_score(LinearRegression(), X, y, cv=kf, scoring="r2")

print("cross_val_score:", scores_cv)
print("Diferença:", scores_manual - scores_cv)

print(f"Score médio:{np.mean(scores_cv):.4f}")

cross_val_score: [0.93113091 0.94750352 0.73564664 0.78433343 0.88053631 0.94724034
 0.81643834 0.86305748 0.71482555 0.86833645]
Diferença: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Score médio:0.8489


## Fique Conectado

- [![YouTube](https://img.icons8.com/?size=40&id=19318&format=png&color=000000)](https://www.youtube.com/@LigaDataScience/videos)  
  Explore nossos vídeos educacionais e webinars sobre ciência de dados, machine learning e inteligência artificial. Inscreva-se para não perder nenhuma atualização!

- [![LinkedIn](https://img.icons8.com/?size=40&id=13930&format=png&color=000000)](https://www.linkedin.com/company/liga-data-science-ufpr/)  
  Siga-nos no LinkedIn para as últimas novidades, oportunidades de carreira e networking profissional no campo da ciência de dados.

- [![Instagram](https://img.icons8.com/?size=40&id=32323&format=png&color=000000)](https://www.instagram.com/ligadatascience/)  
  Confira nosso Instagram para conteúdos dos bastidores, destaques de eventos e o dia a dia da Liga Data Science. Faça parte da nossa jornada!



## Autores

<a href="https://www.linkedin.com/in/eduardopecora/" target="_blank">Eduardo Pecora</a>

## Referências:

* Documentação da biblioteca <a href="https://pandas.pydata.org/docs/">Pandas</a>
* Documentação do método <a href=https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html>cross_val_score</a>
* Documentação do método <a href=https://scikit-learn.org/dev/modules/generated/sklearn.model_selection.KFold.html>KFold</a>
* Documentação do método <a href=https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html>train_test_split</a>


## Log de modificações

| Data | Versão | Modificado por | Descrição |
| -----------| ------- | ---------- | ---------------------------------- |
| 24-04-2024       | 1.0     | Eduardo Pecora    | Inicial               |
| 29-04-2026       | 1.1     | Eduardo Pecora    | Adequações            |
